In [0]:
%run ./00_config

In [0]:
import os

LANDING_DIR = f"{BASE}/landing"
dbutils.fs.mkdirs(LANDING_DIR)

# toate fișierele sursă, ordonate
src_files = sorted(
    os.path.join(root, n)
    for root, _, names in os.walk(READINGS_DIR)
    for n in names
)
print("fișiere sursă:", len(src_files))

def copy_batch(start, end):
    for p in src_files[start:end]:
        dbutils.fs.cp(f"dbfs:{p}" if p.startswith("/Volumes") else p,
                      f"{LANDING_DIR}/{os.path.basename(p)}")
    print("în landing:", len(dbutils.fs.ls(LANDING_DIR)))

copy_batch(21, 40)

In [0]:
df = spark.read.option("header", "true").csv(f"{LANDING_DIR}/LCL-June2015v2_0.csv")
df.show(100)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

raw_schema = StructType([
    StructField("lclid", StringType(), True),
    StructField("stdortou", StringType(), True),
    StructField("datetime", StringType(), True),
    StructField("kwh_raw", StringType(), True),
])

raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")              # fișierele LCL au rând de header
    .schema(raw_schema)
    .load(LANDING_DIR)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp()))
raw.printSchema()



In [0]:
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_readings"
CHECKPOINT = f"{BASE}/_checkpoints/bronze_readings"

q = (raw.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)           # procesează fișierele noi și se oprește
    .toTable(BRONZE_TABLE))
q.awaitTermination()
print("stream terminat, tabel:", BRONZE_TABLE)

In [0]:
bronze = spark.table(BRONZE_TABLE)
print("total rânduri:", bronze.count())
display(bronze.groupBy("_source_file").count().orderBy("_source_file"))
display(bronze.limit(10))

In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT _source_file) AS files FROM bronze_readings;



In [0]:
%sql
SELECT COUNT(*) FROM bronze_readings WHERE lclid = 'LCLid'

In [0]:
%sql
SELECT * FROM bronze_readings LIMIT 10

In [0]:
%sql
SELECT kwh_raw, COUNT(*) FROM bronze_readings
WHERE trim(kwh_raw) NOT RLIKE '^[0-9.]+$' GROUP BY 1


In [0]:
%sql
SELECT _source_file, COUNT(*) AS rows, MIN(_ingested_at) AS first_seen
FROM bronze_readings GROUP BY 1 ORDER BY first_seen

In [0]:
%sql DESCRIBE HISTORY bronze_readings